In [1]:
import os
import sys
import urllib.request
import datetime
import time
import json

In [ ]:
client_id = 'JkDKuSrYnw3mhB2nuJWK'
client_secret = 'c7rVAkosfB'


# [CODE 1] : 실제 검색기(CODE 2 -> CODE 1으로 링크 전달되면 실제 내용 검색)
def getRequestUrl(url):    
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)
    
    try: 
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

# [CODE 2] : 검색 링크 생성기
def getNaverSearch(node, srcText, start, display):    
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)
    
    url = base + node + parameters    
    responseDecode = getRequestUrl(url)   # [CODE 1]
    
    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)

# [CODE 3] : Get 데이터 -> 딕셔너리로 묶음
def getPostData(post, jsonResult, cnt):    
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']
    
    pDate = datetime.datetime.strptime(post['pubDate'],  '%a, %d %b %Y %H:%M:%S +0900')
    pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
    
    jsonResult.append({'cnt':cnt, 'title':title, 'description': description, 
'org_link':org_link,   'link': org_link,   'pDate':pDate})
        

# [CODE 0]
def main():
    node = 'news'   # 크롤링 대상 노드 : 네이버 '뉴스'
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = []

    jsonResponse = getNaverSearch(node, srcText, 1, 100)  # [CODE 2] ## 1: 첫 번째 페이지, 100: 100개(한 번에 가져오는 개수)
    
    total = jsonResponse['total']
 
    while ((jsonResponse != None) and (jsonResponse['display'] != 0)): #응답이 한 개 이상 존재하면 while, 응답 없거나 결과 0개면 멈춤
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)  # [CODE 3]       
        
        start = jsonResponse['start'] + jsonResponse['display'] # start 위치를 페이지마다 변경해서 search나 get 해올 수 있게 함
        if start == 1001: break    # 네이버 뉴스는 1000개까지만 무료 제공됨
        jsonResponse = getNaverSearch(node, srcText, start, 100)  #[CODE 2]       

    print('전체 검색 : %d 건' %total)
    
    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as outfile:
        jsonFile = json.dumps(jsonResult,  indent = 4, sort_keys = True,  ensure_ascii = False)
                        
        outfile.write(jsonFile)
        
    print("가져온 데이터 : %d 건" %(cnt))
    print ('%s_naver_%s.json SAVED' % (srcText, node))
    
if __name__ == '__main__':
    main()


[2025-11-24 10:48:38.220629] Url Request Success
[2025-11-24 10:48:38.338348] Url Request Success
[2025-11-24 10:48:38.454215] Url Request Success
[2025-11-24 10:48:38.589263] Url Request Success
[2025-11-24 10:48:38.717448] Url Request Success
[2025-11-24 10:48:38.847432] Url Request Success
[2025-11-24 10:48:38.991237] Url Request Success
[2025-11-24 10:48:39.146482] Url Request Success
[2025-11-24 10:48:39.312365] Url Request Success
[2025-11-24 10:48:39.462734] Url Request Success
전체 검색 : 71887 건
가져온 데이터 : 1000 건
홍대입구_naver_news.json SAVED
